# Mapping the Workforce: A Social Network Analysis of Job Interconnectivity
## O*NET to Gephi — Pearson Correlation Approach

**Edge Definition**: Two occupations are linked if the Pearson correlation between their skill importance score profiles exceeds 90% similarity
> **Intuition:** If two jobs rank their skills in a similar order of importance, they are connected.
> *"These occupations value the same skills with the same level of priority."*

### Required Files
Download from [O\*NET 30.2](https://www.onetcenter.org/database.html) as **Text/Tab-delimited** (**Note:** Data will be aggregated between these two data files):
- `Occupation Data.txt`
- `Skills.txt`

### Outputs
- `occ_nodes.csv` — Node table for Gephi
- `occ_edges.csv` — Edge table for Gephi

### 1. Imports

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

### 2. Initialization

In [ ]:
OCCUPATION_FILE = "Occupation Data.txt"
SKILLS_FILE     = "Skills.txt"
THRESHOLD       = 0.90   # Pearson correlation cutoff (-1 to 1)
SCALE           = "IM"   # Select only IMPORTANCE values

### 3. Load Data from Files

In [ ]:
def load_data():
    occ = pd.read_csv(OCCUPATION_FILE, delimiter="\t", usecols=["O*NET-SOC Code", "Title"]) # Read data from occupation file
    skills = pd.read_csv(SKILLS_FILE, delimiter="\t",usecols=["O*NET-SOC Code", "Element ID", "Scale ID", "Data Value"]) # Read data from skills file
    return occ, skills

occ, skills = load_data()
occ.head() # Display the first 5 rows

### 4. Build Matrix

In [ ]:
def build_matrix(skills):
    filtered = skills[skills["Scale ID"] == SCALE] # Filter skills where the Scale ID is only IMPORTANCE values

    matrix = filtered.pivot_table(
        index="O*NET-SOC Code",
        columns="Element ID",
        values="Data Value",
        aggfunc="mean"
    ).fillna(0) # Create a matrix by pivoting the skills array so that each skill SOC code and Element ID column has a data value.
    return matrix

matrix = build_matrix(skills) # Build the matrix by calling function
matrix.head() # Retrieve the top 5 entries

### 5. Compute Edges using Pearson Correlation

In [ ]:
def compute_edges(matrix):

    corr_matrix = matrix.T.corr() # First transpose the skill matrix, then compute pearson correlation
    codes = corr_matrix.index.tolist() # Retrieve the columns of the matrix as a list
    edges = [] # Initialize an array for edges

    # Loop through the occupations and check every unique pair of occupations
    for i in range(len(codes)):
        for j in range(i + 1, len(codes)):
            # Retrieve the correlation value between occupation i and j
            corr = corr_matrix.iloc[i, j]

            # Only keep pairs whose pearson correlation is above or equal to set threshold
            if corr >= THRESHOLD:
                # Add the entry to the edge list when condition met
                edges.append({
                    "Source": codes[i],
                    "Target": codes[j],
                    "Weight": round(float(corr), 4)
                })

    edges_df = pd.DataFrame(edges) # Create a dataframe using the edges
    return edges_df

edges_df = compute_edges(matrix) # Compute the edges using the previously created matrix
edges_df.head() # Retrieve the top 5 entries

### 6. Nodes Table

In [ ]:
def build_nodes(occ, edges_df):
    connected = set(edges_df["Source"]).union(set(edges_df["Target"])) # Collect all occupation that appears in the edge dataframe (use set to only collect unique occupations)
    nodes_df = occ[occ["O*NET-SOC Code"].isin(connected)].copy() # Filter occupations from the dataset to only keep the occupations that are within the edges dataframe
    nodes_df = nodes_df.rename(columns={"O*NET-SOC Code": "Id", "Title": "Label"}) # Rename columns as expected by Gephi

    nodes_df["SOC_Major"] = nodes_df["Id"].str[:2] # Extract the first two characters of the occupation code

    soc_labels = { # Dictionary of each occupation code and label (Retrieved from https://www.onetcenter.org/database.html)
        "11": "Management",
        "13": "Business & Financial",
        "15": "Computer & Math",
        "17": "Architecture & Engineering",
        "19": "Life, Physical & Social Science",
        "21": "Community & Social Service",
        "23": "Legal",
        "25": "Education",
        "27": "Arts & Media",
        "29": "Healthcare Practitioners",
        "31": "Healthcare Support",
        "33": "Protective Service",
        "35": "Food Preparation",
        "37": "Building & Grounds",
        "39": "Personal Care",
        "41": "Sales",
        "43": "Office & Admin Support",
        "45": "Farming, Fishing & Forestry",
        "47": "Construction & Extraction",
        "49": "Installation & Repair",
        "51": "Production",
        "53": "Transportation",
        "55": "Military"
    }
    nodes_df["SOC_Group"] = nodes_df["SOC_Major"].map(soc_labels).fillna("Other") # Map each SOC label to each occupation
    return nodes_df # Return the nodes dataframe

nodes_df = build_nodes(occ, edges_df) # Build the nodes
nodes_df.head() # Retrieve the top 5 entries

### 7. Export CSV to Gephi

In [ ]:
def export(nodes_df, edges_df):
    nodes_df.to_csv("occ_nodes.csv", index=False) # Convert the nodes list into a csv file
    edges_df.to_csv("occ_edges.csv", index=False) # Convert the edges list into a csv file

export(nodes_df, edges_df) # Export the nodes and edges list